# Control a Red Pitaya from Python over HTTP

Short examples using `RedPitayaClient`: check status, arm and capture Scope and Spectrum traces, plot/save data, and configure/start/stop Lockbox.

The server must already be running (normally `http://10.0.5.118:8801`). This notebook uses the same instrument session as the web GUI. Run cells from top to bottom. Acquisition cells change acquisition settings; the Lockbox example is opt-in and finishes by unlocking.

## 1. PC setup (Anaconda)

From **Anaconda Prompt**, in this repository:

```text
conda activate pyrpl-env
conda install jupyterlab ipykernel numpy matplotlib
python -m ipykernel install --user --name pyrpl-env --display-name "Python (pyrpl-env)"
jupyter lab examples/http_control_demo.ipynb
```

Select **Python (pyrpl-env)**. The import cell below finds the repository automatically. On another PC, copy `pyrpl_websocket/http_client.py` beside this notebook, or install the repository with `python -m pip install .`.


In [ ]:
import json
import os
from pathlib import Path
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

# Works when Jupyter starts in the repository root or examples directory.
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "pyrpl_websocket" / "http_client.py").is_file():
        sys.path.insert(0, str(folder))
        break
try:
    from pyrpl_websocket.http_client import RedPitayaClient
except ModuleNotFoundError:
    from http_client import RedPitayaClient  # Standalone copy beside the notebook.

BOARD_URL = os.environ.get("PYRPL_HTTP_URL", "http://10.0.5.118:8801")
rp = RedPitayaClient(BOARD_URL, token=os.environ.get("PYRPL_HTTP_TOKEN"),
                    timeout=20, label="Jupyter demo")
print("Connected:", BOARD_URL, "healthy:", rp.health()["ok"])


## 2. Read status and discover available settings

These calls do not require controller ownership. `lock_status` is the latest monitor result; `current_state == "lock_on"` alone does not mean the lock was acquired. Attribute lists include accepted options and limits, which can depend on the Lockbox model.


In [ ]:
print("Lockbox:", rp.lock_status())
print("Controller:", rp.control_status())
print("Scope:", rp.module_state("scope")["state"])
print("Scope actions:", [a["name"] for a in rp.actions("scope")["actions"]])

scope_attributes = {a["name"]: a for a in rp.attributes("scope")["attributes"]}
print("Trigger choices:", scope_attributes["trigger_source"]["options"])


## 3. Arm and capture one Scope trace

`with rp.control()` claims control, renews the lease during the block, and releases it afterward. If a browser currently owns control, use its **Release control** button first. HTTP 409 reports a controller/resource conflict; this notebook does not force takeover.

`single` arms the Scope. The default `immediately` trigger needs no edge. For a CH2 rising-edge capture, set `TRIGGER = "ch2_positive_edge"`, choose the correct `INPUT2`, and set `THRESHOLD` to a level that signal crosses. A missing edge raises `TimeoutError` after the wait. Scope and Spectrum share hardware resources; run these sections sequentially with other measurements stopped.


In [ ]:
INPUT1, INPUT2 = "in1", "out1"
TRIGGER = "immediately"          # Or "ch2_positive_edge".
THRESHOLD = 0.0                   # Volts.
CAPTURE_TIMEOUT = 15.0            # Seconds; increase for slow triggers.

with rp.control():
    rp.action("scope", "stop")
    for attribute, value in {
        "input1": INPUT1, "input2": INPUT2,
        "rolling_mode": False, "trace_average": 1,
        "duration": 0.001, "trigger_delay": 0.0,
        "trigger_source": TRIGGER, "threshold": THRESHOLD,
    }.items():
        rp.set("scope", attribute, value)
    scope_settings = rp.module_state("scope")["state"]  # Actual quantized values.
    previous = rp.scope_frame()   # May be None, or a stale trace from an earlier run.
    try:
        rp.action("scope", "single")
        scope = rp.wait_scope_frame(after=previous, timeout=CAPTURE_TIMEOUT)
    finally:
        rp.action("scope", "stop")  # Also stop if waiting is interrupted or times out.

print(f"Captured {scope.sample_count} samples/channel; sequence {scope.sequence}")
print("Accepted duration:", scope_settings["duration"], "s")


### Plot the Scope data

`scope.channels` contains voltage samples. PWS1 frames carry no time axis or capture settings; the timestamp is a server timestamp, not a hardware trigger time. This plot uses sample index rather than claiming the web GUI's trigger alignment. Settings and frames are separate requests, so avoid simultaneous configuration changes from other clients.


In [ ]:
scope_volts = np.asarray(scope.channels).T  # Shape: samples ? channels.
sample_index = np.arange(scope.sample_count)
fig, ax = plt.subplots(figsize=(9, 3))
for channel in range(scope.channel_count):
    ax.plot(sample_index, scope_volts[:, channel], label=f"CH{channel + 1}")
ax.set(xlabel="Sample index", ylabel="Voltage (V)", title="Single Scope capture")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 4. Capture one Spectrum Analyzer trace

This example uses baseband inputs and a Blackman window. `span` is quantized to supported hardware settings; use the returned frequency axis rather than constructing one. The shared HTTP snapshot contains linear power in **Vpk?**, independent of the web GUI's display-unit selection.


In [ ]:
def wait_spectrum(after=None, timeout=30.0):
    """Wait for a different sequence in the existing server session."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        frame = rp.spectrum_frame(window="blackman")
        if frame is not None and (after is None or frame["sequence"] != after["sequence"]):
            return frame
        time.sleep(0.1)
    raise TimeoutError("No new Spectrum trace; check acquisition state and resource ownership")

with rp.control():
    rp.action("spectrumanalyzer", "stop")
    for attribute, value in {
        "baseband": True, "input1_baseband": "in1", "input2_baseband": "in2",
        "span": 1e6, "window": "blackman", "trace_average": 1,
    }.items():
        rp.set("spectrumanalyzer", attribute, value)
    previous_spectrum = rp.spectrum_frame(window="blackman")
    try:
        rp.action("spectrumanalyzer", "single")
        spectrum = wait_spectrum(after=previous_spectrum)
    finally:
        rp.action("spectrumanalyzer", "stop")

print({key: spectrum[key] for key in ("sequence", "span", "rbw", "unit", "current_avg")})


In [ ]:
frequency_hz = np.asarray(spectrum["x"])
fig, ax = plt.subplots(figsize=(9, 3))
for series in spectrum["series"]:
    # Values are power: 10*log10, not 20*log10. Floor only for plotting zeros.
    power_db = 10 * np.log10(np.maximum(series["values"], 1e-30))
    ax.plot(frequency_hz / 1e3, power_db, label=series["label"])
ax.set(xlabel="Frequency (kHz)", ylabel="Power (dB re 1 Vpk?)", title="Spectrum")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 5. Save both traces on the PC

Files go into `http_demo_data` beside the notebook's working directory. The settings JSON records the Scope settings used, not a trigger-aligned time axis. CSV columns retain linear voltages/powers.


In [ ]:
data_dir = Path("http_demo_data")
data_dir.mkdir(exist_ok=True)
np.savetxt(data_dir / "scope.csv", np.column_stack([sample_index, scope_volts]),
           delimiter=",", header="sample," + ",".join(f"ch{i+1}_V" for i in range(scope.channel_count)), comments="")
(data_dir / "scope_settings.json").write_text(json.dumps(scope_settings, indent=2), encoding="utf-8")
np.savetxt(data_dir / "spectrum.csv",
           np.column_stack([frequency_hz, *(s["values"] for s in spectrum["series"])]),
           delimiter=",", header="frequency_Hz," + ",".join(s["label"] + "_Vpk2" for s in spectrum["series"]), comments="")
print("Saved to", data_dir.resolve())


## 6. Inspect Lockbox parameters

Use names from the running model instead of assuming `output1` exists. This example selects its first input/output and stage 0; change the selections for your setup. The next cell only reads settings.


In [ ]:
def attribute_values(entry):
    return {a["name"]: a["value"] for a in entry["attributes"]}

lockbox = rp.lockbox()
print("Model:", lockbox["classname"], "status:", lockbox["state"])
print("Inputs:", [entry["name"] for entry in lockbox["inputs"]])
print("Outputs:", [entry["name"] for entry in lockbox["outputs"]])
for index, stage in enumerate(lockbox["sequence"]):
    print("Stage", index, attribute_values(stage))

INPUT_NAME = lockbox["inputs"][0]["name"]
OUTPUT_NAME = lockbox["outputs"][0]["name"]
STAGE = 0
print("Selected input:", attribute_values(lockbox["inputs"][0]))
print("Selected output:", attribute_values(lockbox["outputs"][0]))


## 7. Set parameters, start a lock, observe status, then unlock

Set `RUN_LOCKBOX_DEMO = True` only when you want to run this section. Edit the gains and setpoint for your experiment; they are examples, not a tuning prescription. Setpoint units and gain conventions depend on the selected model and calibration.

The example uses manual P/I gains (`assisted_design=False`), enables the selected output in stage 0, and starts the **entire configured locking sequence**. It leaves other stages and output routing as configured. The loop observes cached monitor status; the `is_locked` action below requests a fresh check. Starting a sequence does not guarantee lock acquisition.

The `finally` block unlocks after this short demonstration, including on interruption. Parameter changes remain saved. Releasing HTTP control by itself would **not** unlock the hardware.


In [ ]:
RUN_LOCKBOX_DEMO = False
P_GAIN = 0.1
I_GAIN = 100.0
SETPOINT = 0.0
OBSERVE_SECONDS = 3.0

if RUN_LOCKBOX_DEMO:
    with rp.control():
        try:
            rp.lockbox_action("unlock")
            rp.set_lockbox("auto_lock", False)
            rp.set_lockbox_output(OUTPUT_NAME, "assisted_design", False)
            rp.set_lockbox_output(OUTPUT_NAME, "p", P_GAIN)
            rp.set_lockbox_output(OUTPUT_NAME, "i", I_GAIN)
            # To change the physical input, for example:
            # rp.set_lockbox_input(INPUT_NAME, "input_signal", "in1")
            rp.set_stage(STAGE, "input", INPUT_NAME)
            rp.set_stage(STAGE, "setpoint", SETPOINT)
            rp.set_stage(STAGE, "gain_factor", 1.0)
            rp.set_stage_output(STAGE, OUTPUT_NAME, "lock_on", True)
            rp.lockbox_action("lock")
            deadline = time.monotonic() + OBSERVE_SECONDS
            while time.monotonic() < deadline:
                print(rp.lock_status())
                time.sleep(0.5)
            print("Fresh lock check:", rp.lockbox_action("is_locked")["state"])
        finally:
            rp.lockbox_action("unlock")
    print("After demo:", rp.lock_status())
else:
    print("Lockbox demo skipped. Set RUN_LOCKBOX_DEMO=True to apply parameters and run it.")


### Individual commands for interactive use

Run the commands you need in a new cell:

```python
with rp.control():
    rp.lockbox_action("lock")     # Leaves the configured sequence running.

print(rp.lock_status())           # Read status without claiming control.

with rp.control():
    rp.lockbox_action("unlock")   # Explicitly stop feedback.
```

To save the current configuration as a named profile, or transfer one as JSON:

```python
with rp.control():
    saved = rp.create_profile("My experiment")  # Name must be unique.
profile_id = saved["active"]["id"]
Path("experiment.json").write_text(json.dumps(rp.export_profile(profile_id), indent=2), encoding="utf-8")
# Loading a profile applies settings and stops acquisition/Lockbox:
# with rp.control():
#     rp.load_profile(profile_id)
```

Full commands, return formats, and error descriptions: [HTTP API guide](../docs/source/http_api.rst). A timeout does not undo a write; inspect status before retrying. The examples use latest shared frames, not lossless streaming; if the service restarts during capture, rerun that capture cell.
